In [78]:
import requests
import simplejson
import pandas as pd
import sys
from copy import deepcopy

In [79]:
root_url = "https://sc-data-dev.emsl.pnl.gov"
samples = requests.get(f"{root_url}/sample?page=1&per_page=1000").json()['samples']

In [80]:
sample_ecoregions = requests.get(f"{root_url}/elastic/metadata").json()

In [81]:
toc_tns = []
pages = sys.maxsize
page = 1
while page <= pages:
    res = requests.get(f"{root_url}/elastic/TOC_TN?page={page}&per_page=100&data=true").json()
    records = res['data']
    pages = res['pages']
    page += 1
    toc_tns.extend(records)

In [82]:
toc_tns_dict = {f"{tt['proposal_id']}_{tt['sampling_set']}_{tt['core_section']}": tt for tt in toc_tns}

In [84]:
sample_ecoregions_dict = {se['sampling_data']['id']: se['sampling_data'] for se in sample_ecoregions}

In [85]:
monet_samples = []
for sample in samples:
    sampling_ecoregion = sample_ecoregions_dict[sample['id']]
    has_toc_tn_top = f"{sample['proposal_id']}_{sample['sampling_set']}_TOP" in toc_tns_dict.keys()
    has_toc_tn_bottom = f"{sample['proposal_id']}_{sample['sampling_set']}_BTM" in toc_tns_dict.keys()
    if has_toc_tn_top:
        toc_tn_top = toc_tns_dict[f"{sample['proposal_id']}_{sample['sampling_set']}_TOP"]
    if has_toc_tn_bottom:
        toc_tn_bottom = toc_tns_dict[f"{sample['proposal_id']}_{sample['sampling_set']}_BTM"]

    monet_sample = {
        "ber_data_source": "MONET",
        "coordinates": {
            "latitude": float(sample['latitude']),
            "longitude": float(sample['longitude']),
            "altitude": None,
            "depth": None,
            "elevation": {
                "numeric_value": float(sample['elevation']['value']),
                "unit": sample['elevation']['unit']
            }
        },
        "entity_type": [
            f"sample"
        ],
        "description": None,
        "id": sample['id'],
        "name": f"MONet Core {sample['proposal_id']}_{sample['sampling_set']}",
        "alt_ids": None,
        "alt_names": None,
        "part_of_collection": None,
        "uri": "https://sc-data.emsl.pnnl.gov/monet",
        "properties": [
            {
                "attribute": {
                    "id": "MIXS:0000332",
                    "label": "soil_type"
                },
                "raw_value": sample['soil_metadata']['soil_type']
            },
            {
                "attribute": {
                    "id": "MIXS:0000011",
                    "label": "collection_date"
                },
                "raw_value": sample['collection_date']
            },
            {
                "attribute": {
                    "label": "ecoregion"
                },
                "raw_value": sampling_ecoregion['eco_info']['ecoregion']
            }
        ]
    }
    monet_sample_top = deepcopy(monet_sample)
    monet_sample_bottom = deepcopy(monet_sample)
    monet_sample_top['name'] = f"{monet_sample_top['name']}_TOP"
    monet_sample_bottom['name'] = f"{monet_sample_bottom['name']}_BOTTOM"
    if has_toc_tn_top:
        monet_sample_top['properties'].extend([
            {
                "attribute": {
                    "label": "toc_avg"
                },
                "raw_value": f"{toc_tn_top['toc_avg']} {toc_tn_top['toc_unit']}",
                "unit": toc_tn_top['toc_unit'],
                "numeric_value": toc_tn_top['toc_avg']
            },{
                "attribute": {
                    "label": "tn_avg"
                },
                "raw_value": f"{toc_tn_top['tn_avg']} {toc_tn_top['tn_unit']}",
                "unit": toc_tn_top['tn_unit'],
                "numeric_value": toc_tn_top['tn_avg']
            }

        ])
    if has_toc_tn_bottom:
        monet_sample_bottom['properties'].extend([
            {
                "attribute": {
                    "label": "toc_avg"
                },
                "raw_value": f"{toc_tn_bottom['toc_avg']} {toc_tn_bottom['toc_unit']}",
                "unit": toc_tn_bottom['toc_unit'],
                "numeric_value": toc_tn_bottom['toc_avg']
            },{
                "attribute": {
                    "label": "tn_avg"
                },
                "raw_value": f"{toc_tn_bottom['tn_avg']} {toc_tn_bottom['tn_unit']}",
                "unit": toc_tn_bottom['tn_unit'],
                "numeric_value": toc_tn_bottom['tn_avg']
            }

        ])
    monet_samples.extend([monet_sample_top, monet_sample_bottom])


In [86]:
emsl_samples_df = pd.read_json('emsl_samples.json')

In [87]:
emsl_samples = emsl_samples_df.to_dict(orient='records')

In [88]:
emsl_samples.extend(monet_samples)

In [89]:
with open('data/emsl_00001.json', 'w') as f:
    f.write(simplejson.dumps(emsl_samples, ignore_nan=True))

In [92]:
with open('/Users/dett541/dev/bertron-schema/src/sample_data/valid/monet-example.json', 'w') as f:
    f.write(simplejson.dumps(emsl_samples[0], ignore_nan=True))
with open('/Users/dett541/dev/bertron-schema/src/sample_data/valid/emsl-example.json', 'w') as f:
    f.write(simplejson.dumps(emsl_samples[-1], ignore_nan=True))